# Week 2: Shoreline Extraction & Dataset Assembly
### Lagos Barrier Island Shoreline Change Detection Project

**Purpose:** Threshold MNDWI using Otsu's method, vectorize into shoreline polygons, filter out speckle noise, and assemble the final 2017–2026 shoreline dataset. Re-runs acquisition + preprocessing here for a self-contained notebook, then does the extraction work unique to this stage.

Output: `../data/lagos_shoreline_dataset_2017_2026.gpkg`

## 1. Authenticate & Initialize Earth Engine

In [ ]:
import ee

ee.Authenticate(
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/devstorage.full_control",
        "https://www.googleapis.com/auth/drive",
    ],
)
ee.Initialize()


## 2. Imports & AOI

In [ ]:
from pipeline_utils import (
    get_best_scene, mask_clouds_scl, compute_mndwi,
    extract_water_mask, mask_to_shoreline, clean_shoreline_vectors,
    process_year,
)
import ee
import geemap
import geopandas as gpd
import os

aoi = ee.Geometry.Rectangle([3.30, 6.38, 3.55, 6.48])


## 3. Run the full pipeline across all years (2017–2026)

Uses `process_year` from `pipeline_utils.py`, which chains acquisition through noise cleanup for a single year — the exact same logic validated cell-by-cell earlier, now consolidated.

In [ ]:
years = list(range(2017, 2026))
shoreline_dataset = {}
final_thresholds = {}

for yr in years:
    result = process_year(yr, aoi)
    if result is not None:
        shoreline_dataset[yr], final_thresholds[yr] = result
        print(f"{yr}: processed and cleaned (threshold={final_thresholds[yr]:.4f})")

print(f"\nFinal dataset: {len(shoreline_dataset)} / {len(years)} years complete.")


## 4. Visual QA — extracted shoreline for a sample year

In [ ]:
sample_year = 2020

scene = get_best_scene(sample_year, aoi)
masked = mask_clouds_scl(scene)
mndwi_img = compute_mndwi(masked)

Map3 = geemap.Map(center=[6.43, 3.42], zoom=12)
Map3.addLayer(
    mndwi_img,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    f"True color {sample_year}",
)
Map3.addLayer(
    shoreline_dataset[sample_year],
    {"color": "red"},
    f"Extracted shoreline {sample_year}",
)
Map3.addLayer(aoi, {}, "AOI", opacity=0.3)
Map3


## 5. Merge all years into one dataset and export locally

Pulls data directly via `.getInfo()` (proven reliable throughout this project) rather than Earth Engine's asynchronous export-to-Drive mechanism, which requires additional auth scopes and hit repeated errors during development. Saved as GeoPackage (`.gpkg`) rather than Shapefile, since GeoPackage has no field-name-length limit and is a single file rather than a multi-file set.

In [ ]:
tagged_collections = [
    fc.map(lambda f, yr=yr: f.set("year", yr))
    for yr, fc in shoreline_dataset.items()
]

full_shoreline_dataset = ee.FeatureCollection(tagged_collections).flatten()

print(f"Total features in combined dataset: {full_shoreline_dataset.size().getInfo()}")

geojson = full_shoreline_dataset.getInfo()

gdf = gpd.GeoDataFrame.from_features(geojson["features"])
gdf = gdf.set_crs(epsg=4326)

output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/lagos_shoreline_dataset_2017_2026.gpkg"
gdf.to_file(output_path, driver="GPKG")

print(f"Saved to {output_path}")
